In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

### Load SPX option quotes

In [2]:
data_dir = Path("spx_2023q2")

april = pd.read_csv(data_dir / "spx_eod_202304.txt", skipinitialspace=True)
may = pd.read_csv(data_dir / "spx_eod_202305.txt", skipinitialspace=True)
june = pd.read_csv(data_dir / "spx_eod_202306.txt", skipinitialspace=True)

### Filter option contracts

Keep contracts with positive bid, ask, volume, and IV; valid bid/ask spreads; strikes within 20% of the underlying; and 2-180 DTE. ATM means the strike is within 1% of the underlying price; ITM and OTM depend on option type. `MID_PRICE` is the average of bid and ask.

In [3]:
options = pd.concat([april, may, june], ignore_index=True)
options.columns = options.columns.str.strip("[] ")

numeric_columns = [
    "DTE", "UNDERLYING_LAST", "STRIKE",
    "C_BID", "C_ASK", "C_VOLUME", "C_IV",
    "P_BID", "P_ASK", "P_VOLUME", "P_IV",
]
options[numeric_columns] = (
    options[numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)
print(f"Loaded {len(options):,} strike/date rows")

Loaded 472,279 strike/date rows


In [ ]:
## strikes from 80% to 120% of the underlying ##
STRIKE_BAND = 0.20  
## Within 1% of the underlying is ATM ##
ATM_BAND = 0.01     

strike_to_spot = options["STRIKE"] / options["UNDERLYING_LAST"]
dte_mask = options["DTE"].between(2, 180)

common_mask = (
    options["UNDERLYING_LAST"].gt(0)
    & options["STRIKE"].gt(0)
    & strike_to_spot.between(1 - STRIKE_BAND, 1 + STRIKE_BAND)
    & dte_mask
)
common_columns = ["QUOTE_DATE", "EXPIRE_DATE", "DTE", "UNDERLYING_LAST", "STRIKE"]


def select_contracts(side):
    side_columns = [f"{side}_{name}" for name in ("BID", "ASK", "VOLUME", "IV")]
    bid, ask, volume, iv = (options[column] for column in side_columns)
    valid = (
        common_mask
        & bid.gt(0)
        & ask.gt(0)
        & bid.le(ask)
        & volume.gt(0)
        & iv.gt(0)
    )
    contracts = options.loc[valid, common_columns + side_columns].copy()
    contracts.rename(
        columns=dict(zip(side_columns, ("BID", "ASK", "VOLUME", "IV"))),
        inplace=True,
    )
    contracts.insert(2, "OPTION_TYPE", "CALL" if side == "C" else "PUT")
    contracts["MID_PRICE"] = (contracts["BID"] + contracts["ASK"]) / 2
    contracts["STRIKE_TO_SPOT"] = contracts["STRIKE"] / contracts["UNDERLYING_LAST"]
    ratio = contracts["STRIKE_TO_SPOT"]
    is_atm = ratio.between(1 - ATM_BAND, 1 + ATM_BAND)
    is_itm = ratio.lt(1 - ATM_BAND) if side == "C" else ratio.gt(1 + ATM_BAND)
    contracts["MONEYNESS"] = np.select([is_atm, is_itm], ["ATM", "ITM"], default="OTM")
    return contracts


calls = select_contracts("C")
puts = select_contracts("P")
filtered_options = pd.concat([calls, puts], ignore_index=True)

print(f"Filtered contracts: {len(filtered_options):,}")
print(filtered_options.groupby(["OPTION_TYPE", "MONEYNESS"]).size())

Filtered contracts: 409,623
OPTION_TYPE  MONEYNESS
CALL         ATM           28159
             ITM          101318
             OTM           67868
PUT          ATM           27435
             ITM           21085
             OTM          163758
dtype: int64


In [5]:
if filtered_options.empty:
    raise ValueError("No contracts passed the filters. Check the source data or filter settings.")

daily_volume = filtered_options.groupby("QUOTE_DATE")["VOLUME"].sum()
max_volume_date = daily_volume.idxmax()
highest_volume_date = filtered_options.loc[
    filtered_options["QUOTE_DATE"].eq(max_volume_date)
].copy()
highest_volume_call = calls.loc[calls["VOLUME"].eq(calls["VOLUME"].max())].copy()
highest_volume_put = puts.loc[puts["VOLUME"].eq(puts["VOLUME"].max())].copy()

print(f"Highest filtered-volume date: {max_volume_date} ({daily_volume.max():,.0f} contracts)")

cleaned_folder = Path("cleaned files")
cleaned_folder.mkdir(exist_ok=True)
filtered_options.to_csv(cleaned_folder / "spx_options_filtered_2023q2.csv", index=False)
highest_volume_date.to_csv(cleaned_folder / "highest_volume_day.csv", index=False)
highest_volume_call.to_csv(cleaned_folder / "highest_volume_call.csv", index=False)
highest_volume_put.to_csv(cleaned_folder / "highest_volume_put.csv", index=False)

Highest filtered-volume date: 2023-06-30 (800,847 contracts)
